# MINI assembly qualification replay

Checks the exact tangent contraction against the merged reference, runs the full suite and repeats the unchanged 23-solve locking campaign. The 2026-10-23 Edge Colab execution passed; see docs/MIXED_MINI_ASSEMBLY_RESULTS.md for results, evidence and limitations. Timings are specific to the paired runtime. No patient data is used.

Evidence export runs after failed setup or qualification, and offers the archive before raising the final failure gate.


In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, platform, subprocess, sys, traceback, zipfile

CODE_COMMIT = "584785c32caadd3c6bc42fef712b3762789add85"
REFERENCE_COMMIT = "b2dacf79a8d0cdb93fbd34e5d78b19d4ee06d534"
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
ROOT = Path("/content") / ("mini_assembly_" + STAMP)
ROOT.mkdir()
REPO = ROOT / "CardiMech"
EVIDENCE = ROOT / "evidence"
EVIDENCE.mkdir()
STAGES = []
ENV = {**os.environ, "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}
def stage(name, command, cwd=None, required=True):
    started = datetime.datetime.now(datetime.timezone.utc)
    result = subprocess.run(command, cwd=cwd, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (EVIDENCE / (name + ".log")).write_text(result.stdout)
    STAGES.append({"name": name, "command": command, "returncode": result.returncode,
                   "started_utc": started.isoformat(),
                   "elapsed_seconds": (datetime.datetime.now(datetime.timezone.utc)-started).total_seconds()})
    (EVIDENCE / "stages.json").write_text(json.dumps(STAGES, indent=2))
    print(name, "returncode", result.returncode, flush=True)
    print(result.stdout[-4000:], flush=True)
    if required:
        result.check_returncode()
    return result

failure = None
try:
    # Colab's distribution can lack ensurepip; use the PyPI virtualenv seeder.
    # Avoid naming any diagnostic file profile.py, which shadows the stdlib module.
    stage("virtualenv-install", [sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
    stage("venv", [sys.executable, "-m", "virtualenv", str(ROOT / "venv")])
    PY = str(ROOT / "venv/bin/python")
    stage("clone", ["git", "clone", "https://github.com/Virelion-Biotech/Virelion-CardiMech.git", str(REPO)])
    stage("checkout", ["git", "checkout", CODE_COMMIT], cwd=REPO)
    stage("install", [PY, "-m", "pip", "install", "-q", "-e", str(REPO) + "[dev,validation,spatial]"])
    INFER = ROOT / "CardiInfer"
    stage("infer-clone", ["git", "clone", "https://github.com/Virelion-Biotech/Virelion-CardiInfer.git", str(INFER)])
    stage("infer-checkout", ["git", "checkout", "15a7bd5bf4c23784245b361c017a376572766254"], cwd=INFER)
    stage("infer-install", [PY, "-m", "pip", "install", "-q", "-e", str(INFER)])
    stage("dependencies", [PY, "-m", "pip", "freeze"])
    stage("lint", [PY, "-m", "ruff", "check", "src", "tests", "scripts"], cwd=REPO, required=False)
    stage("format", [PY, "-m", "ruff", "format", "--check", "src", "tests", "scripts"], cwd=REPO, required=False)
    stage("suite", [PY, "-m", "pytest", "-q", "--junitxml=" + str(EVIDENCE / "junit.xml"),
                    "--cov=cardimech", "--cov-fail-under=80"], cwd=REPO, required=False)
    stage("comparison", [PY, "scripts/compare_mini_assembly.py", "--output",
                         str(EVIDENCE / "comparison")], cwd=REPO, required=False)
    stage("qualification", [PY, "scripts/verify_mixed_locking.py", "--output",
                            str(EVIDENCE / "qualification")], cwd=REPO, required=False)
except Exception:
    failure = traceback.format_exc()
    (EVIDENCE / "failure.txt").write_text(failure)
    print(failure, flush=True)

manifest = {"candidate_code_commit": CODE_COMMIT, "reference_commit": REFERENCE_COMMIT,
            "python": platform.python_version(), "stages": STAGES,
            "scope": "synthetic software and numerical regression; no patient accuracy qualification",
            "all_commands_passed": failure is None and all(s["returncode"] == 0 for s in STAGES),
            "failure": failure,
            "source_sha256": {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest()
                              for p in sorted((REPO / "src").rglob("*.py"))}}
(EVIDENCE / "manifest.json").write_text(json.dumps(manifest, indent=2))
archive = ROOT / "mixed_mini_assembly_evidence.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in EVIDENCE.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(EVIDENCE))
receipt = {"path": str(archive), "bytes": archive.stat().st_size,
           "sha256": hashlib.sha256(archive.read_bytes()).hexdigest()}
print("RECEIPT", json.dumps(receipt), flush=True)
print("ALL_COMMANDS_PASSED", manifest["all_commands_passed"], flush=True)
# Offer the archive on both success and failure, before raising the final gate.
# A failed setup or regression must not prevent retrieval of its own diagnostics.
from google.colab import files
try:
    files.download(str(archive))
except Exception:
    print("Browser download could not be offered; archive remains at", archive)
    print(traceback.format_exc())
assert manifest["all_commands_passed"], "Failed replay: retrieve the evidence archive and diagnose before merge."
